In [1]:
import mdtraj as md
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic_2d
import pyvista as pv
from sklearn.decomposition import PCA
from Bio import PDB
import plotly.graph_objects as go
import pandas as pd
from scipy.spatial.distance import cdist

In [ ]:

TRAJ_PATH = '/data/bpti_md/protein/bpti_all_trajectory_NCCO.npy'
PDB_DIR = '/data/esmdiff_target/bpti_kinetic_cluster/bpti_5'
PDB_FILES = [f"{PDB_DIR}/bpti_{i}.pdb" for i in range(1, 6)]

In [4]:
# --- 1. Trajectory 데이터 로드 및 에너지 계산 ---
print("Trajectory 데이터 로드 중...")
traj_data = np.load(TRAJ_PATH)  # (N, 58, 4, 3)
N = traj_data.shape[0]

# PCA용 (N, 696) 평탄화
traj_flat = traj_data.reshape(N, -1)

Trajectory 데이터 로드 중...


In [6]:
def get_ncco_coords(pdb_file):
    """
    PDB 파일에서 58개 잔기의 N, CA, C, O 원자 좌표만 추출합니다.
    """
    t = md.load(pdb_file)
    # BPTI 58개 잔기 확인 및 N, CA, C, O 선택
    selection = t.topology.select("resSeq 1 to 58 and (name N or name CA or name C or name O)")
    
    # MDTraj는 nm 단위이므로 Angstrom으로 변환 (논문 기준)
    coords = t.xyz[0, selection, :]
    
    # (58, 4, 3) 형태로 reshape
    return coords.reshape(58, 4, 3)

def kabsch_alignment(target, reference):
    """
    Kabsch 알고리즘을 사용하여 target 구조를 reference 구조에 정렬합니다.
    (Translation 및 Rotation 제거)
    """
    # 1. 중심 맞추기 (Translation 제거)
    target_center = target.mean(axis=0)
    ref_center = reference.mean(axis=0)
    
    t_centered = target - target_center
    r_centered = reference - ref_center
    
    # 2. 회전 행렬 계산 (SVD)
    u, s, vh = np.linalg.svd(np.dot(t_centered.T, r_centered))
    rotation_matrix = np.dot(u, vh)
    
    # 3. 좌표 변환
    aligned_coords = np.dot(t_centered, rotation_matrix) + ref_center
    return aligned_coords

# --- [준비 단계] Theseus 방식 가중치 및 평균 구조 추출 ---
def get_theseus_weights(traj_data):
    """
    traj_data: (N, 232, 3) - 이미 Theseus로 정렬된 42만개 데이터
    """
    # 1. 평균 구조 (Reference Mean)
    mean_ref = np.mean(traj_data, axis=0)  # (232, 3)
    
    # 2. 원자별 분산 계산 (232개 원자 각각의 흔들림 측정)
    # 각 원자의 (x, y, z) 편차 제곱의 평균
    deviations = traj_data - mean_ref
    variances = np.mean(np.sum(deviations**2, axis=-1), axis=0) # (232,)
    
    # 3. 가중치 계산 (ML 방식: 1/variance)
    # 수치적 안정을 위해 아주 작은 값을 더해줌
    weights = 1.0 / (variances + 1e-6)
    weights /= np.sum(weights) # 가중치 정규화
    
    return mean_ref, weights

# --- [정렬 단계] 가중치 적용 Kabsch 알고리즘 ---
def weighted_kabsch_alignment(target, reference, weights):
    """
    target: (232, 3), reference: (232, 3), weights: (232,)
    """
    w = weights.reshape(-1, 1)
    
    # 1. 가중 무게중심 이동
    target_wcm = np.sum(target * w, axis=0) / np.sum(w)
    ref_wcm = np.sum(reference * w, axis=0) / np.sum(w)
    
    t_centered = target - target_wcm
    r_centered = reference - ref_wcm
    
    # 2. 가중치가 적용된 공분산 행렬 및 SVD
    W = np.diag(weights)
    covariance_matrix = np.dot(np.dot(t_centered.T, W), r_centered)
    u, s, vh = np.linalg.svd(covariance_matrix)
    
    # 3. 회전 및 변환
    rotation_matrix = np.dot(u, vh)
    aligned_coords = np.dot(t_centered, rotation_matrix) + ref_wcm
    return aligned_coords

In [ ]:
# --- 1. Theseus 가중치 및 평균 구조 추출 ---
# traj_data shape: (N, 58, 4, 3) -> (N, 232, 3)로 변환하여 처리
print("Theseus 가중치 및 평균 구조 추출 중...")
traj_data_232 = traj_data.reshape(N, 232, 3)
mean_ref, atom_weights = get_theseus_weights(traj_data_232)

# --- 2. PCA 실행 (전체 트래젝토리 기준) ---
print("PCA 실행 및 학습 중...")
# traj_flat shape: (N, 696)
pca = PCA(n_components=2)
coords_pca = pca.fit_transform(traj_flat)

# --- 3. 5개 PDB 파일 분석 (정렬, PCA 투영, 에너지 계산) ---
new_pcs = []
new_es = []

print(f"\n{len(PDB_FILES)}개 PDB 파일 분석 시작...")

for i, pdb_path in enumerate(PDB_FILES):
    if not os.path.exists(pdb_path):
        print(f"파일 없음: {pdb_path}")
        continue
    
    # 3-1. NCCO 좌표 추출 (58, 4, 3) -> (232, 3) 및 Angstrom 변환 포함됨
    raw_coords = get_ncco_coords(pdb_path).reshape(232, 3)
    
    # 3-2. Theseus 방식 가중치 정렬 (Weighted Kabsch)
    # 기존 데이터의 평균(mean_ref)과 가중치(atom_weights)를 기준으로 신규 PDB 정렬
    aligned_coords = weighted_kabsch_alignment(raw_coords, mean_ref, atom_weights)

    
    # 3-4. PCA 공간 투영 (Projection)
    # 정렬된 좌표를 평탄화 (1, 696) 하여 기존 PCA 모델에 적용
    aligned_flat = aligned_coords.reshape(1, -1)
    pca_proj = pca.transform(aligned_flat)
    new_pcs.append(pca_proj[0])
    
    print(f"[PDB {i+1}] {os.path.basename(pdb_path)} - 에너지: {energy:.2f}, PC1: {pca_proj[0][0]:.2f}, PC2: {pca_proj[0][1]:.2f}")

# 결과 데이터를 numpy 배열로 변환
new_pcs = np.array(new_pcs)
new_es = np.array(new_es)

print("\n--- 모든 분석 완료 ---")
print(f"신규 PDB PC 좌표 집합:\n{new_pcs}")
print(f"신규 PDB 에너지 집합:\n{new_es}")

Theseus 가중치 및 평균 구조 추출 중...
PCA 실행 및 학습 중...

5개 PDB 파일 분석 시작...
[PDB 1] bpti_1.pdb - 에너지: -107932.01, PC1: -0.09, PC2: 0.19
[PDB 2] bpti_2.pdb - 에너지: -107592.24, PC1: -0.32, PC2: -1.59
[PDB 3] bpti_3.pdb - 에너지: -106097.55, PC1: -0.20, PC2: -1.54
[PDB 4] bpti_4.pdb - 에너지: -107408.87, PC1: -0.16, PC2: 0.71
[PDB 5] bpti_5.pdb - 에너지: -107507.07, PC1: -0.29, PC2: 0.28

--- 모든 분석 완료 ---
신규 PDB PC 좌표 집합:
[[-0.08869267  0.19440362]
 [-0.31524944 -1.5913571 ]
 [-0.20317459 -1.5361985 ]
 [-0.16495037  0.70974106]
 [-0.28813934  0.28332633]]
신규 PDB 에너지 집합:
[-107932.00995569 -107592.24313061 -106097.55034265 -107408.86615418
 -107507.07499067]


In [ ]:
from Bio import PDB
import numpy as np

# --- 4. 멀티 모델 PDB 분석 (100개 모델 + O 좌표 패치) ---
multi_model_pdb = "/esmdiff_var/slm/models/residual_120K/bpti/step25_eps1e-05_N100_20251217-113252/bpti.pdb"
print(f"\n멀티 모델 PDB({multi_model_pdb}) 분석 시작...")

parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure('ensemble', multi_model_pdb)

ensemble_pcs = []
ensemble_energies = [] # 에너지 저장 리스트

print(f"\n앙상블 모델 분석 및 DFIRE 에너지 계산 시작...")

for model in structure:
    model_coords = []
    for chain in model:
        residues = [r for r in chain if PDB.is_aa(r)]
        for i, residue in enumerate(residues):
            try:
                n, ca, c = residue['N'].get_coord(), residue['CA'].get_coord(), residue['C'].get_coord()
                try:
                    o = residue['O'].get_coord()
                except KeyError:
                    # O 좌표 패치 로직
                    u_ca_c = (c - ca) / np.linalg.norm(c - ca)
                    u_ca_n = (n - ca) / np.linalg.norm(n - ca)
                    o = c + 1.231 * ( (u_ca_c + u_ca_n) / np.linalg.norm(u_ca_c + u_ca_n) )
                model_coords.append([n, ca, c, o])
            except KeyError: continue
    
    if len(model_coords) != 58: continue
    raw_coords = np.array(model_coords).reshape(232, 3) * 0.1
    aligned_coords = weighted_kabsch_alignment(raw_coords, mean_ref, atom_weights)
    
    # PCA 투영
    pca_proj = pca.transform(aligned_coords.reshape(1, -1))
    ensemble_pcs.append(pca_proj[0])

ensemble_pcs = np.array(ensemble_pcs)


멀티 모델 PDB(/esmdiff_var/slm/models/residual_120K/bpti/step25_eps1e-05_N100_20251217-113252/bpti.pdb) 분석 시작...

앙상블 모델 분석 및 DFIRE 에너지 계산 시작...
신규 PDB 에너지 집합:
[-108212.46160624 -108293.90960612 -108346.51658564 -108464.57419389
 -108291.30087376 -108421.68421132 -108095.52575448 -108391.32078757
 -108083.11617682 -108104.63222644 -108286.90170062 -108048.53445189
 -108220.02486863 -108265.63359041 -108244.87375598 -108123.22681255
 -108250.16046004 -108115.30594622 -108448.49722067 -108274.33934119
 -107987.65215269 -108582.26765559 -108327.13197299 -108050.42414074
 -108139.38818696 -107995.36233682 -108288.7206368  -108146.11938497
 -108297.15048275 -108129.38432617 -108445.6650228  -107887.0202909
 -108068.60025802 -108353.60572594 -108157.56925627 -108055.2538837
 -108147.61597165 -107907.47989237 -108203.57439798 -108249.46262839
 -108180.87254859 -108615.67234312 -107959.28346533 -108275.72148957
 -108333.57447257 -108382.11526963 -108287.45594368 -108242.11758481
 -108179.85469743

In [ ]:
from Bio import PDB
import numpy as np

# --- 4. 멀티 모델 PDB 분석 (100개 모델 + O 좌표 패치) ---
multi_model_pdb = "/bpti_esmdiff_baseline.pdb"
print(f"\n멀티 모델 PDB({multi_model_pdb}) 분석 시작...")

parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure('ensemble', multi_model_pdb)

ensemble_pcs2 = []
ensemble_energies2 = [] # 에너지 저장 리스트

print(f"\n앙상블 모델 분석 및 DFIRE 에너지 계산 시작...")

for model in structure:
    model_coords = []
    for chain in model:
        residues = [r for r in chain if PDB.is_aa(r)]
        for i, residue in enumerate(residues):
            try:
                n, ca, c = residue['N'].get_coord(), residue['CA'].get_coord(), residue['C'].get_coord()
                try:
                    o = residue['O'].get_coord()
                except KeyError:
                    # O 좌표 패치 로직
                    u_ca_c = (c - ca) / np.linalg.norm(c - ca)
                    u_ca_n = (n - ca) / np.linalg.norm(n - ca)
                    o = c + 1.231 * ( (u_ca_c + u_ca_n) / np.linalg.norm(u_ca_c + u_ca_n) )
                model_coords.append([n, ca, c, o])
            except KeyError: continue
    
    if len(model_coords) != 58: continue
    raw_coords = np.array(model_coords).reshape(232, 3) * 0.1
    aligned_coords = weighted_kabsch_alignment(raw_coords, mean_ref, atom_weights)

    
    # PCA 투영
    pca_proj = pca.transform(aligned_coords.reshape(1, -1))
    ensemble_pcs2.append(pca_proj[0])

ensemble_pcs2 = np.array(ensemble_pcs2)



멀티 모델 PDB(/bpti_esmdiff_baseline.pdb) 분석 시작...

앙상블 모델 분석 및 DFIRE 에너지 계산 시작...
신규 PDB 에너지 집합:
[-108410.63786887 -107991.91685581 -108370.93928253 -108028.44376757
 -107741.09202963 -108322.65422869 -108151.51638541 -108613.53574998
 -108001.0270405  -108280.67048817 -107868.29714042 -107980.04165753
 -107789.00366455 -108187.8265033  -108013.53972553 -108418.75587165
 -108302.09188977 -108361.67680146 -108603.73403802 -108625.21580833
 -108376.56009181 -108384.52927638 -108221.48140927 -108372.68989984
 -108172.59160111 -108284.21357241 -108189.15819002 -108217.16256928
 -108375.82661988 -108443.01255922 -108305.81604909 -107451.75120662
 -108342.15279986 -108325.98870992 -108657.29636168 -108338.45809992
 -108145.06741415 -108495.69410537 -108071.89788385 -108521.86563145
 -108159.03501098 -108671.84654423 -108043.76385145 -108868.56472823
 -107883.83381804 -107384.16411095 -108658.53269965 -108492.45841247
 -108186.58710836 -107914.34347892 -108680.3888437  -107776.94018281
 -108378

In [10]:
import numpy as np
import plotly.graph_objects as go

# --- 1. 자유 에너지 지형도 재계산 (마스킹 추가) ---
grid_size = 80
hist, x_edges, y_edges = np.histogram2d(
    coords_pca[:, 0], coords_pca[:, 1], bins=grid_size, density=True
)

# 데이터가 없는 곳(hist == 0)을 마스킹하기 위해 복사본 생성
Z_kt = -np.log(hist + 1e-10) # 일단 로그 계산
Z_kt -= np.min(Z_kt)
Z_kt = Z_kt.T # Plotly Surface 대응을 위한 전치

# [핵심 수정] 데이터가 없는 구역(hist가 매우 낮은 곳)을 NaN으로 처리하여 렌더링에서 제외
# 이렇게 하면 위쪽이 잘린 '천장' 대신 데이터 경계선에서 지형이 자연스럽게 끊깁니다.
mask = hist.T < (np.max(hist) * 0.001) # 전체 밀도의 0.1% 미만인 곳은 무시
Z_kt[mask] = np.nan

X, Y = np.meshgrid((x_edges[:-1] + x_edges[1:]) / 2, (y_edges[:-1] + y_edges[1:]) / 2)

# --- 2. PDB 점들의 Z값 매핑 ---
def get_surface_z(pc_x, pc_y, x_edges, y_edges, z_values):
    ix = np.searchsorted(x_edges, pc_x) - 1
    iy = np.searchsorted(y_edges, pc_y) - 1
    ix = np.clip(ix, 0, z_values.shape[1] - 1)
    iy = np.clip(iy, 0, z_values.shape[0] - 1)
    return z_values[iy, ix]

new_es_on_surface = [
    get_surface_z(new_pcs[i, 0], new_pcs[i, 1], x_edges, y_edges, Z_kt) 
    for i in range(len(new_pcs))
]

In [ ]:
# --- 5. 앙상블 점들의 Z값 매핑 ---
ensemble_es_on_surface = [
    get_surface_z(ensemble_pcs[i, 0], ensemble_pcs[i, 1], x_edges, y_edges, Z_kt) 
    for i in range(len(ensemble_pcs))
]
ensemble_es_on_surface2 = [
    get_surface_z(ensemble_pcs2[i, 0], ensemble_pcs2[i, 1], x_edges, y_edges, Z_kt) 
    for i in range(len(ensemble_pcs2))
]

# --- 6. Plotly 시각화 업데이트 ---
fig = go.Figure()

# A. Surface (기존 마스킹 지형도)
fig.add_trace(go.Surface(
    z=Z_kt, x=X, y=Y, 
    colorscale='Viridis', reversescale=True, opacity=0.1,
    colorbar_title='Density',
    contours=dict(z=dict(show=True, usecolormap=True, project_z=True, highlightcolor="white"))
))

# B. 신규 100개 모델 (파란색 점)
fig.add_trace(go.Scatter3d(
    x=ensemble_pcs[:, 0], y=ensemble_pcs[:, 1], z=ensemble_es_on_surface,
    mode='markers',
    marker=dict(
        size=5, 
        color='blue', 
        opacity=0.6,
        line=dict(color='white', width=1)
    ),
    name='Ours (100 Models)'
))

fig.add_trace(go.Scatter3d(
    x=ensemble_pcs2[:, 0], y=ensemble_pcs2[:, 1], z=ensemble_es_on_surface2,
    mode='markers',
    marker=dict(
        size=5, 
        color='red', 
        opacity=0.6,
        line=dict(color='white', width=1)
    ),
    name='ESMDiff (100 Models)'
))

# C. 기존 5개 레퍼런스 PDB (빨간색 다이아몬드)
fig.add_trace(go.Scatter3d(
    x=new_pcs[:, 0], y=new_pcs[:, 1], z=new_es_on_surface,
    mode='markers+text',
    marker=dict(
        size=10, 
        color='red', 
        symbol='diamond', 
        line=dict(color='black', width=2)
    ),
    text=[f"PDB {i+1}" for i in range(len(new_pcs))],
    textposition="top center",
    name='Reference PDBs'
))


def get_synced_camera(degree, distance=4.0, elevation=2.2):
    """
    degree: 원하는 회전 각도 (0~360)
    distance: 카메라와 물체 사이의 거리 (r)
    elevation: 카메라의 높이 (z축 위치)
    """
    rad = np.radians(degree)
    
    return dict(
        eye=dict(
            x=distance * np.cos(rad),
            y=distance * np.sin(rad),
            z=elevation
        ),
        center=dict(x=0, y=0, z=0),
        up=dict(x=0, y=0, z=1)
    )

# 1. 원하는 시점 정의 (예: 45도 방향에서 바라보기)
sync_view = get_synced_camera(degree=180, distance=0)

fig.update_layout(
    title='BPTI Density Landscape: Reference vs Ensemble',
    scene=dict(
        xaxis_title='PC1', yaxis_title='PC2', 
        zaxis_title='Density',
        camera=sync_view,
        zaxis=dict(range=[0, np.nanmax(Z_kt)],
            showticklabels=False,
            showgrid=True,
            zeroline=False  ) 
    ),
    width=1000, height=800,
    legend=dict(x=0, y=1)
)

fig.show()